<a href="https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Feature vector

The prediction features are:

- `content_age_days` — age of the content in days.
- `days_since_last_update` — days since the content was last updated.
- `impressions_90d` — impressions observed over the available 90-day window.
- `avg_position` — average search position.
- `ctr` — click-through rate.

The label `is_declining_label` is kept separate from the feature vector.

Missing GSC-related values are treated as unavailable evidence rather than zero performance. Categorical identifiers such as client and content IDs are not used as model predictors.

In [1]:
feature_fields = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

print("Feature vector:")
for feature in feature_fields:
    print("✓", feature)

print("\nNumber of features:", len(feature_fields))
print("Label kept separate: is_declining_label")

Feature vector:
✓ content_age_days
✓ days_since_last_update
✓ impressions_90d
✓ avg_position
✓ ctr

Number of features: 5
Label kept separate: is_declining_label


### Feature notes

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| `content_age_days` | Age of the content | Kept as missing/unavailable if not known | Yes |
| `days_since_last_update` | Days since last update | Kept as missing/unavailable if not known | Yes |
| `impressions_90d` | Observed impressions in the available 90-day window | Missing means unavailable GSC evidence, not zero | Yes |
| `avg_position` | Observed average search position | Missing means unavailable GSC evidence | Yes |
| `ctr` | Observed click-through rate | Missing means unavailable GSC evidence | Yes |

All five features are intended to represent information available at the decision moment. No future outcome is intentionally used as a predictor.

In [2]:
feature_notes = {
    "content_age_days": "Content age; available before prediction.",
    "days_since_last_update": "Days since last update; available before prediction.",
    "impressions_90d": "90-day observed impressions; missing means unavailable evidence.",
    "avg_position": "Observed average position; missing means unavailable evidence.",
    "ctr": "Observed click-through rate; missing means unavailable evidence."
}

for feature, note in feature_notes.items():
    print(f"{feature}: {note}")

print("\nAll selected features are pre-decision features.")

content_age_days: Content age; available before prediction.
days_since_last_update: Days since last update; available before prediction.
impressions_90d: 90-day observed impressions; missing means unavailable evidence.
avg_position: Observed average position; missing means unavailable evidence.
ctr: Observed click-through rate; missing means unavailable evidence.

All selected features are pre-decision features.


### Leakage hunt

I checked for three main leakage risks:

1. **Label-derived fields** — `is_declining_label` and `trend_direction` must not enter the feature vector.
2. **Future information** — future-window performance must not be used because it would not be available at the decision moment.
3. **Identifiers/product flags** — identifiers such as `content_id` and `client_id` are not predictive evidence and can create privacy or grouping issues.

The selected five features are based on information observed before the prediction decision. The checks below confirm that the known label-derived and future fields are excluded.

In [3]:
feature_vector = set(feature_fields)

forbidden_fields = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "future_performance",
    "content_id",
    "client_id"
}

leakage_found = feature_vector.intersection(forbidden_fields)

print("Leakage / privacy check")
print("=" * 40)

print("Selected features:")
for feature in sorted(feature_vector):
    print("✓", feature)

print("\nForbidden fields checked:")
for field in sorted(forbidden_fields):
    print("✓", field)

print("\nFields accidentally included:", leakage_found)

if len(leakage_found) == 0:
    print("RESULT: No checked leakage fields are in the feature vector.")
else:
    print("RESULT: Review required.")


Leakage / privacy check
Selected features:
✓ avg_position
✓ content_age_days
✓ ctr
✓ days_since_last_update
✓ impressions_90d

Forbidden fields checked:
✓ client_id
✓ content_id
✓ future_performance
✓ is_declining_label
✓ trend_direction
✓ trend_pct

Fields accidentally included: set()
RESULT: No checked leakage fields are in the feature vector.


### Excluded fields

- `is_declining_label` — this is the prediction label, so using it as a feature would directly leak the answer.
- `trend_direction` — directly describes the observed outcome and would leak the label.
- `trend_pct` — related to the supplied trend outcome and therefore excluded.
- `future_performance` — future information would not be available at the decision moment.
- `content_id` — identifier only; not useful as predictive evidence and may create memorization/grouping issues.
- `client_id` — grouping/context information; excluded as a model predictor to reduce privacy and client-specific leakage risk.
- Product/future outcome flags — excluded when they depend on information unavailable before the decision.

In [4]:
excluded_fields = {
    "is_declining_label": "Prediction label; direct leakage.",
    "trend_direction": "Outcome-derived field; direct leakage.",
    "trend_pct": "Related to the supplied trend outcome.",
    "future_performance": "Future information unavailable at decision time.",
    "content_id": "Identifier only; not predictive evidence.",
    "client_id": "Client grouping/context; excluded as a predictor."
}

print("Excluded fields and reasons")
print("=" * 40)

for field, reason in excluded_fields.items():
    print(f"{field}: {reason}")

Excluded fields and reasons
is_declining_label: Prediction label; direct leakage.
trend_direction: Outcome-derived field; direct leakage.
trend_pct: Related to the supplied trend outcome.
future_performance: Future information unavailable at decision time.
content_id: Identifier only; not predictive evidence.
client_id: Client grouping/context; excluded as a predictor.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.